# Appendix K: Sleep and Duty Cycles Across Scale

Executable verification code extracted from Appendix K of *Our Fractal Universe*.
Reproduces all numerical claims: mammalian sleep-mass power law fitting,
the complete 16-level duty cycle table, cycle-period-vs-mass regression,
and generation of the three diagnostic plots.

Plot outputs are saved to `issue34_plots/`.


## Setup and Imports


In [ ]:
#!/usr/bin/env python3
"""
M4: Sleep/Duty Cycles -- Full Expansion to ALL 16 Levels

Expands the Appendix I Section 4 analysis from the existing ~9 scale coverage
to 16 levels: Proton through Universe, including City and Nation (which Chris
specifically requested).

For each level we tabulate:
  - System mass (kg)
  - Characteristic cycle period (seconds)
  - Duty fraction (active fraction or rest fraction, as appropriate)
  - Mechanism / description

Generates three plots:
  1. duty_fraction_vs_mass.png  -- rest fraction vs system mass (log-log)
  2. cycle_period_vs_mass.png   -- cycle period vs system mass (log-log)
  3. cross_scale_comparison.png -- visual comparison chart

Sources are cited inline. All timescales and fractions use best available
peer-reviewed or well-established values.

Usage:
    python sleep_duty_cycles.py
"""

import os
import math
import numpy as np

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# ============================================================
# OUTPUT DIRECTORY
# ============================================================
SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))
PLOT_DIR = os.path.join(SCRIPT_DIR, 'issue34_plots')
RESEARCH_DIR = os.path.join(SCRIPT_DIR, 'issue34_research')
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(RESEARCH_DIR, exist_ok=True)


## DATA: 16 Scale Levels

Complete duty cycle data across all 16 levels, from proton to observable universe.


In [ ]:
# ============================================================
# DATA: 16 levels with duty cycle information
# ============================================================
# Each entry: (name, mass_kg, cycle_period_s, rest_fraction,
#              cycle_description, source)
#
# "rest_fraction" = fraction of time in quiescent/dormant/inactive state
# For levels where multiple cycles exist, we pick the most prominent one.

DUTY_CYCLE_DATA = [
    # 1. PROTON
    # Delta(1232) resonance excitation/decay (~5x10^-24 s).
    # Proton spends overwhelming fraction in ground state.
    ("Proton",
     1.67e-27,           # mass (kg), CODATA 2022
     5.0e-24,            # cycle period: Delta resonance lifetime (s)
     1e-6,               # rest fraction: vanishingly small; set floor for plotting
     "Delta(1232) resonance excitation/decay (~5x10^-24 s); "
     "proton spends overwhelming fraction in ground state",
     "PDG 2022; width ~120 MeV => tau ~ 5e-24 s"),

    # 2. ATOM (Hydrogen)
    # Electron excitation/relaxation: Ly-alpha lifetime ~1.6 ns.
    ("Atom (H)",
     1.67e-27,           # mass (kg) -- hydrogen atom ~ proton mass
     1.6e-9,             # cycle period: Ly-alpha spontaneous emission (s)
     1e-8,               # rest fraction: excited state occupation in typical conditions
     "Electron excitation/relaxation; Ly-alpha lifetime ~1.6 ns; "
     "atom spends >99.999999% in ground state under typical conditions",
     "NIST ASD; Wiese & Fuhr 2009"),

    # 3. RIBOSOME
    # Translation pausing: ~15% of time in paused states.
    ("Ribosome",
     2.50e-21,           # mass (kg), BioNumbers BNID 102320
     60.0,               # cycle period: full translation cycle for typical protein (s)
     0.15,               # rest fraction: ~15% time paused (Ingolia et al. 2011)
     "Translation elongation with pausing; ~5-6 codons/s with "
     "~15% time in paused states for co-translational folding",
     "Ingolia et al. 2011; Rodnina 2016; PMC5311927"),

    # 4. BACTERIUM (E. coli)
    # Growth-dormancy cycle: ~30% in lag/stationary/persister states.
    ("Bacterium (E. coli)",
     1.00e-15,           # mass (kg), BioNumbers
     7200.0,             # cycle period: lag + growth cycle ~2 hr (s)
     0.30,               # rest fraction: ~30% in lag/stationary/persister states
     "Growth-dormancy cycle; lag phase (~1-2 hr), exponential growth, "
     "stationary phase; persister fraction 0.01-1%",
     "Balaban et al. 2004; Levin-Reisman et al. 2017; PMC3837759"),

    # 5. C. ELEGANS
    # Lethargus: ~2 hr quiescence before each of 4 larval molts.
    ("C. elegans",
     1.00e-9,            # mass (kg), NCBI
     36000.0,            # cycle period: ~10 hr intermolt + lethargus (s)
     0.20,               # rest fraction: ~2 hr lethargus per ~10 hr cycle
     "Lethargus: ~2 hr quiescence before each of 4 larval molts; "
     "satisfies all behavioral criteria for sleep (Raizen et al. 2008)",
     "Raizen et al. 2008; Van Buskirk & Bhatt 2007; PMC3735717"),

    # 6. HUMAN
    ("Human",
     70.0,               # mass (kg)
     86400.0,            # cycle period: 24 hr circadian cycle (s)
     0.33,               # rest fraction: 8/24 = 0.33
     "Circadian sleep-wake cycle; ~8 hr sleep per 24 hr day; "
     "includes NREM and REM stages",
     "Consensus; Siegel 2005"),

    # 7. CITY
    ("City",
     1.00e10,            # mass (kg), Bettencourt 2007; West 2017
     86400.0,            # cycle period: 24 hr day-night cycle (s)
     0.30,               # rest fraction: ~7 hr low-activity (2 AM - 9 AM)
     "Urban activity cycle: peak activity 9 AM - 9 PM, "
     "reduced to ~5-10% between 2-5 AM; night economy ~4% of GDP (UK 2022)",
     "WEF 2024; World Cities Culture Forum; Bettencourt 2007"),

    # 8. NATION
    ("Nation",
     1.00e13,            # mass (kg): ~10 million people x 70 kg + infrastructure
     1.7e9,              # cycle period: Kondratieff wave ~54 yr (s)
     0.25,               # rest fraction: ~25% in stagnation/recession phase
     "Kondratieff wave: ~50-54 yr economic long cycle; "
     "~25% 'winter' (recession/depression) phase; "
     "US business cycle: 16% contraction (NBER post-WWII)",
     "Kondratiev 1925; NBER; Modelski & Thompson 1996"),

    # 9. EARTH
    ("Earth",
     5.97e24,            # mass (kg), IAU 2015
     86400.0,            # cycle period: 24 hr rotation (s)
     0.50,               # rest fraction: ~50% dark hemisphere at any time
     "Diurnal cycle: 24 hr rotation; biosphere receives energy during day. "
     "Also: seasonal (1 yr, 25% winter), Milankovitch (~100 kyr, 80% glacial)",
     "IAU; Francois et al. 1998"),

    # 10. SUN
    ("Sun",
     1.99e30,            # mass (kg), IAU 2015
     3.47e8,             # cycle period: 11 yr sunspot cycle (s)
     0.03,               # rest fraction: ~3% in grand minima
     "11-yr sunspot cycle with ~0.1% luminosity variation; "
     "grand minima (Maunder Minimum) ~1-5% of time; "
     "CME frequency varies 10x across cycle",
     "Usoskin et al. 2007; Lean 2000"),

    # 11. SOLAR SYSTEM
    ("Solar System",
     2.00e30,            # mass (kg), IAU 2015
     2.0e15,             # cycle period: ~63 Myr full vertical oscillation (s)
     0.15,               # rest fraction: ~15% near galactic midplane (denser phase)
     "Vertical oscillation through galactic plane: ~30-37 Myr half-period; "
     "midplane crossings correlate with enhanced cometary bombardment",
     "Rampino & Stothers 1984; Bahcall & Bahcall 1985"),

    # 12. OPEN CLUSTER
    ("Open Cluster",
     5.00e32,            # mass (kg), Appendix A
     3.15e15,            # cycle period: ~100 Myr relaxation timescale (s)
     0.90,               # rest fraction: ~90% passive after star formation ceases
     "Star formation episode ~10^6-10^7 yr then passive evolution; "
     "dynamical relaxation time ~10^8 yr; half-life 150-800 Myr",
     "Binney & Tremaine 2008"),

    # 13. LOCAL BUBBLE
    ("Local Bubble",
     1.00e34,            # mass (kg), Appendix A
     3.5e13,             # cycle period: ~1.1 Myr between supernovae (s)
     0.70,               # rest fraction: ~70% coasting/quiet between SN events
     "Expansion driven by ~15 supernovae over 14 Myr; "
     "~1.1 Myr between events; currently coasting at ~6 km/s",
     "Zucker et al. 2022; Breitschwerdt et al. 2016"),

    # 14. MILKY WAY
    ("Milky Way",
     1.50e42,            # mass (kg), Bland-Hawthorn 2016
     3.15e15,            # cycle period: ~100 Myr AGN episode timescale (s)
     0.95,               # rest fraction: ~95% quiescent (1-5% AGN active)
     "AGN duty cycle: 1-10% active fraction; episodes ~10^7-10^8 yr; "
     "galactic rotation ~225 Myr; SF rate varies 2-3x",
     "Martini 2004; Bland-Hawthorn & Gerhard 2016"),

    # 15. VIRGO SUPERCLUSTER
    ("Virgo Supercluster",
     1.00e45,            # mass (kg), Appendix A
     3.15e16,            # cycle period: ~1 Gyr between significant mergers (s)
     0.92,               # rest fraction: ~92% quiescent between merger events
     "Assembly over ~13 Gyr; merger events on ~Gyr timescale; "
     "cosmic expansion (model-inferred) since z~0.5 slows collapse; "
     "no well-defined periodic cycle",
     "Einasto et al. 2019; Tully et al. 2014"),

    # 16. OBSERVABLE UNIVERSE
    ("Observable Universe",
     1.50e53,            # mass (kg), Planck 2018
     4.35e17,            # cycle period: age of universe ~13.8 Gyr (s)
     0.29,               # "rest" fraction: ~4/13.8 Gyr accelerating phase
     "Single inferred transition: deceleration (0-9.8 Gyr) to acceleration "
     "(9.8-13.8 Gyr); not a periodic cycle; "
     "recent DESI data suggests possible re-deceleration",
     "Planck 2018; Riess et al. 1998; DESI 2024"),
]


## Mammalian Sleep Data, Power-Law Fits, and Diagnostic Plots


In [ ]:
# ============================================================
# MAMMALIAN SLEEP DATA (for the power-law fit)
# ============================================================
MAMMAL_DATA = [
    ("Little brown bat",   0.008,  19.9),
    ("Opossum",            1.5,    18.0),
    ("Rat",                0.3,    12.6),
    ("Cat",                3.3,    12.5),
    ("Rabbit",             2.5,     8.4),
    ("Human",             70.0,     8.0),
    ("Cow",              500.0,     3.9),
    ("Horse",            450.0,     2.9),
    ("Giraffe",         1200.0,     1.9),
    ("African elephant", 4000.0,    3.3),
]


def fit_mammalian_sleep():
    """Fit power law: sleep_fraction = k * M^alpha."""
    masses = np.array([d[1] for d in MAMMAL_DATA])
    fracs = np.array([d[2] / 24.0 for d in MAMMAL_DATA])
    log_m = np.log10(masses)
    log_f = np.log10(fracs)
    coeffs = np.polyfit(log_m, log_f, 1)
    alpha = coeffs[0]
    log_k = coeffs[1]
    k = 10**log_k
    predicted = np.polyval(coeffs, log_m)
    ss_res = np.sum((log_f - predicted)**2)
    ss_tot = np.sum((log_f - np.mean(log_f))**2)
    r_sq = 1.0 - ss_res / ss_tot
    return alpha, k, r_sq


def main():
    print("=" * 80)
    print("M4: SLEEP / DUTY CYCLES -- FULL 16-LEVEL EXPANSION")
    print("=" * 80)

    # -- Mammalian power law fit --
    alpha, k, r_sq = fit_mammalian_sleep()
    print(f"\nMAMMALIAN SLEEP-MASS POWER LAW:")
    print(f"  sleep_fraction = {k:.4f} * M^({alpha:.4f})")
    print(f"  R^2 = {r_sq:.4f}")
    print(f"  Literature range: alpha ~ -0.12 to -0.15 (larger datasets)")

    # -- Full duty cycle table --
    print(f"\n{'='*80}")
    print(f"COMPLETE DUTY CYCLE TABLE (16 LEVELS)")
    print(f"{'='*80}\n")

    header = (f"{'Level':<24s} {'Mass (kg)':>12s} {'Cycle (s)':>12s} "
              f"{'Rest Frac':>10s} {'Pred Frac':>10s} {'Pred/Obs':>10s}")
    print(header)
    print("-" * 80)

    names_all = []
    masses_all = []
    rest_fracs_all = []
    cycle_periods_all = []

    for (name, mass, period, rest_frac, desc, source) in DUTY_CYCLE_DATA:
        pred_frac = min(k * mass**alpha, 1.0)
        if pred_frac < 1e-30:
            pred_frac = 1e-30
        ratio = rest_frac / pred_frac if pred_frac > 0 else float('inf')
        print(f"{name:<24s} {mass:>12.2e} {period:>12.2e} "
              f"{rest_frac:>10.4f} {pred_frac:>10.2e} {ratio:>10.2e}")

        names_all.append(name)
        masses_all.append(mass)
        rest_fracs_all.append(rest_frac)
        cycle_periods_all.append(period)

    masses_all = np.array(masses_all)
    rest_fracs_all = np.array(rest_fracs_all)
    cycle_periods_all = np.array(cycle_periods_all)

    # -- Cycle period vs mass power-law fit --
    log_masses = np.log10(masses_all)
    log_periods = np.log10(cycle_periods_all)

    coeffs_t = np.polyfit(log_masses, log_periods, 1)
    beta = coeffs_t[0]
    log_c = coeffs_t[1]
    c_coeff = 10**log_c

    pred_log_t = np.polyval(coeffs_t, log_masses)
    ss_res_t = np.sum((log_periods - pred_log_t)**2)
    ss_tot_t = np.sum((log_periods - np.mean(log_periods))**2)
    r_sq_t = 1.0 - ss_res_t / ss_tot_t

    print(f"\n  Fit: cycle_period = {c_coeff:.4e} * M^({beta:.4f})")
    print(f"  R^2 = {r_sq_t:.4f}")

    # -- Key findings --
    print(f"\n{'='*80}")
    print(f"KEY FINDINGS")
    print(f"{'='*80}")
    print()
    print("  1. REST-ACTIVITY CYCLING IS UNIVERSAL across all 16 levels.")
    print("  2. MAMMALIAN POWER LAW DOES NOT EXTRAPOLATE to cosmic scales.")
    print("  3. CITY AND NATION show genuine duty cycles.")
    print("  4. CYCLE PERIOD broadly increases with mass (beta ~ {:.2f})".format(beta))
    print(f"     but with enormous scatter (R^2 = {r_sq_t:.2f}).")
    print("  5. OPEN QUESTION: Whether ubiquity reflects a deep principle")
    print("     or generic physics of energy-dissipating systems remains unresolved.")

    # ============================================================
    # PLOTS
    # ============================================================
    domain_colors = {
        'Proton': '#9467bd', 'Atom (H)': '#9467bd',
        'Ribosome': '#2ca02c', 'Bacterium (E. coli)': '#2ca02c',
        'C. elegans': '#1f77b4', 'Human': '#1f77b4',
        'City': '#ff7f0e', 'Nation': '#ff7f0e',
        'Earth': '#d62728',
        'Sun': '#e377c2', 'Solar System': '#e377c2',
        'Open Cluster': '#8c564b', 'Local Bubble': '#8c564b',
        'Milky Way': '#7f7f7f', 'Virgo Supercluster': '#7f7f7f',
        'Observable Universe': '#17becf',
    }
    colors = [domain_colors.get(n, '#333333') for n in names_all]

    # -- Plot 1: Rest fraction vs mass --
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.scatter(masses_all, rest_fracs_all, c=colors, s=120, zorder=5,
               edgecolors='black', linewidths=0.8)
    for i, name in enumerate(names_all):
        offset_x, offset_y = 0.15, 0.08
        if name == 'Atom (H)': offset_y = -0.15
        elif name == 'Solar System': offset_y = -0.15
        elif name == 'Nation': offset_x, offset_y = 0.2, -0.12
        ax.annotate(name, (masses_all[i], rest_fracs_all[i]),
                    xytext=(offset_x, offset_y), textcoords='offset fontsize',
                    fontsize=7.5, ha='left', va='center')
    m_range = np.logspace(-28, 55, 500)
    pred_line = np.clip(k * m_range**alpha, 1e-30, 1.0)
    ax.plot(m_range, pred_line, 'r--', linewidth=1.5, alpha=0.5,
            label=f'Mammalian power law: f = {k:.3f}M^{{{alpha:.3f}}} '
                  f'(R^2={r_sq:.2f})')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(1e-29, 1e55); ax.set_ylim(1e-9, 2.0)
    ax.set_xlabel('System Mass (kg)', fontsize=12)
    ax.set_ylabel('Rest / Quiescent Fraction', fontsize=12)
    ax.set_title('Sleep/Duty Cycle Rest Fraction vs System Mass\n'
                 'Across 16 Organizational Levels', fontsize=13)
    ax.legend(loc='lower left', fontsize=9)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, 'duty_fraction_vs_mass.png'),
                dpi=150, bbox_inches='tight')
    plt.close()

    # -- Plot 2: Cycle period vs mass --
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.scatter(masses_all, cycle_periods_all, c=colors, s=120, zorder=5,
               edgecolors='black', linewidths=0.8)
    for i, name in enumerate(names_all):
        offset_x, offset_y = 0.15, 0.08
        if name == 'Atom (H)': offset_y = -0.15
        elif name == 'Earth': offset_y = -0.15
        elif name == 'City': offset_x, offset_y = 0.2, -0.12
        ax.annotate(name, (masses_all[i], cycle_periods_all[i]),
                    xytext=(offset_x, offset_y), textcoords='offset fontsize',
                    fontsize=7.5, ha='left', va='center')
    m_fit = np.logspace(-28, 55, 500)
    t_fit = c_coeff * m_fit**beta
    ax.plot(m_fit, t_fit, 'r--', linewidth=1.5, alpha=0.5,
            label=f'Power law: T = {c_coeff:.1e} M^{{{beta:.3f}}} '
                  f'(R^2={r_sq_t:.2f})')
    ref_times = [
        (1e-9, '1 nanosecond'), (1.0, '1 second'), (3600, '1 hour'),
        (86400, '1 day'), (3.15e7, '1 year'), (3.15e10, '1 kyr'),
        (3.15e13, '1 Myr'), (3.15e16, '1 Gyr'),
    ]
    for t_ref, label in ref_times:
        ax.axhline(y=t_ref, color='gray', linestyle=':', linewidth=0.5,
                    alpha=0.4)
        ax.text(1e54, t_ref, label, fontsize=6, va='bottom', ha='right',
                color='gray', alpha=0.7)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(1e-29, 1e55); ax.set_ylim(1e-25, 1e19)
    ax.set_xlabel('System Mass (kg)', fontsize=12)
    ax.set_ylabel('Characteristic Cycle Period (s)', fontsize=12)
    ax.set_title('Duty Cycle Period vs System Mass\n'
                 'Across 16 Organizational Levels', fontsize=13)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, 'cycle_period_vs_mass.png'),
                dpi=150, bbox_inches='tight')
    plt.close()

    # -- Plot 3: Cross-scale comparison --
    fig, ax = plt.subplots(figsize=(16, 10))
    y_positions = np.arange(len(names_all))
    active_fracs = 1.0 - rest_fracs_all
    ax.barh(y_positions, active_fracs, height=0.6,
            color=colors, edgecolor='black', linewidth=0.5,
            label='Active fraction')
    ax.barh(y_positions, rest_fracs_all, height=0.6, left=active_fracs,
            color='#cccccc', edgecolor='black', linewidth=0.5,
            label='Rest/quiescent fraction')
    for i, name in enumerate(names_all):
        if active_fracs[i] > 0.15:
            ax.text(active_fracs[i]/2, y_positions[i],
                    f'{active_fracs[i]*100:.0f}% active',
                    ha='center', va='center', fontsize=7,
                    fontweight='bold', color='white')
        if rest_fracs_all[i] > 0.08:
            ax.text(active_fracs[i] + rest_fracs_all[i]/2, y_positions[i],
                    f'{rest_fracs_all[i]*100:.0f}% rest',
                    ha='center', va='center', fontsize=7, color='#333333')
    ax.set_yticks(y_positions)
    ax.set_yticklabels([f'{n}\n(M={m:.0e} kg)' for n, m in
                        zip(names_all, masses_all)], fontsize=8)
    ax.set_xlabel('Fraction of Cycle', fontsize=12)
    ax.set_title('Cross-Scale Comparison: Active vs Rest/Quiescent Fractions\n'
                 'Across 16 Organizational Levels (Proton to Universe)',
                 fontsize=13)
    ax.set_xlim(0, 1)
    ax.legend(loc='lower right', fontsize=9)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, 'cross_scale_comparison.png'),
                dpi=150, bbox_inches='tight')
    plt.close()

    print("\nPlots saved to:", PLOT_DIR)
    print("M4 COMPLETE")


if __name__ == '__main__':
    main()
